# 체계DB ERLCLIF05 자동화

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

Settings

In [2]:
BASE_DIR = os.getcwd()
위험률테이블_경로 = os.path.join(BASE_DIR, "inputs", "riskrate.txt")
위험률코드매핑_경로 = os.path.join(BASE_DIR, "inputs", "code_map.txt")
위험룰구분자매핑_경로 = os.path.join(BASE_DIR, "inputs", "factor_map.txt")

회사 위험률 테이블

In [3]:
df_riskrate = pd.read_csv(위험률테이블_경로, sep="\t", dtype=str)
print(df_riskrate.shape)
df_riskrate.head()

(528, 17)


,위험률코드,위험률명,이력일자,산출단위,위험률,구분자1,구분자1 유효값,구분자2,구분자2 유효값,구분자3,구분자3 유효값,구분자4,구분자4 유효값,구분자5,구분자5 유효값,구분자6,구분자6 유효값
0,MR000001,질병사망,2026-03-01,1,0.00090,MC000002,1,MC000003,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MR000001,질병사망,2026-03-01,1,0.00112,MC000002,1,MC000003,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MR000001,질병사망,2026-03-01,1,0.00134,MC000002,1,MC000003,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MR000001,질병사망,2026-03-01,1,0.00156,MC000002,1,MC000003,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MR000001,질병사망,2026-03-01,1,0.00178,MC000002,1,MC000003,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


위험률코드 매핑 테이블

In [4]:
df_riskrate_code_map = pd.read_csv(위험률코드매핑_경로, sep="\t", dtype=str)
print(df_riskrate_code_map.shape)
df_riskrate_code_map.head()

(2, 6)


,위험률코드_개발원,위험구분1,위험구분2,위험구분3,위험구분4,위험률코드_회사
0,100000,49,50,21,NaN,MR000004
1,200000,49,50,NaN,NaN,MR000001


위험률구분자 매핑 테이블

In [5]:
df_factor_map = pd.read_csv(위험룰구분자매핑_경로, sep="\t", dtype=str)
print(df_factor_map.shape)
df_factor_map.head()

(71, 4)


,위험구분_회사,위험구분유효값_회사,위험구분_개발원,위험구분유효값_개발원
0,MC000001,1,21,1
1,MC000001,2,21,2
2,MC000001,3,21,3
3,MC000002,1,49,1
4,MC000002,2,49,2


테이블 JOIN

In [6]:
df_joined = pd.merge(df_riskrate, df_riskrate_code_map, left_on="위험률코드", right_on="위험률코드_회사", how="left")
df_joined.drop(columns=["위험률코드"], inplace=True)
print(df_joined.shape)
df_joined.head()

(528, 22)


,위험률명,이력일자,산출단위,위험률,구분자1,구분자1 유효값,구분자2,구분자2 유효값,구분자3,구분자3 유효값,...,구분자5,구분자5 유효값,구분자6,구분자6 유효값,위험률코드_개발원,위험구분1,위험구분2,위험구분3,위험구분4,위험률코드_회사
0,질병사망,2026-03-01,1,0.00090,MC000002,1,MC000003,15,NaN,NaN,...,NaN,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001
1,질병사망,2026-03-01,1,0.00112,MC000002,1,MC000003,16,NaN,NaN,...,NaN,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001
2,질병사망,2026-03-01,1,0.00134,MC000002,1,MC000003,17,NaN,NaN,...,NaN,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001
3,질병사망,2026-03-01,1,0.00156,MC000002,1,MC000003,18,NaN,NaN,...,NaN,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001
4,질병사망,2026-03-01,1,0.00178,MC000002,1,MC000003,19,NaN,NaN,...,NaN,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001


위험률구분자 및 유효값 변환

In [7]:
구분자_컬럼명_list = ['구분자1', '구분자2', '구분자3', '구분자4', '구분자5', '구분자6']
구분자유효값_컬럼명_list = ['구분자1 유효값', '구분자2 유효값', '구분자3 유효값', '구분자4 유효값', '구분자5 유효값', '구분자6 유효값']


df_joined["위험률조건_회사"] = [
    set(
        (row[c1], row[c2])
        for c1, c2 in zip(구분자_컬럼명_list, 구분자유효값_컬럼명_list)
        if pd.notna(row[c1]) and pd.notna(row[c2])
    )
    for row in df_joined.to_dict("records")
]

print(df_joined.shape)
df_joined.head()

(528, 23)


,위험률명,이력일자,산출단위,위험률,구분자1,구분자1 유효값,구분자2,구분자2 유효값,구분자3,구분자3 유효값,...,구분자5 유효값,구분자6,구분자6 유효값,위험률코드_개발원,위험구분1,위험구분2,위험구분3,위험구분4,위험률코드_회사,위험률조건_회사
0,질병사망,2026-03-01,1,0.00090,MC000002,1,MC000003,15,NaN,NaN,...,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 15), (MC000002, 1)}"
1,질병사망,2026-03-01,1,0.00112,MC000002,1,MC000003,16,NaN,NaN,...,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 16), (MC000002, 1)}"
2,질병사망,2026-03-01,1,0.00134,MC000002,1,MC000003,17,NaN,NaN,...,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 17), (MC000002, 1)}"
3,질병사망,2026-03-01,1,0.00156,MC000002,1,MC000003,18,NaN,NaN,...,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 18), (MC000002, 1)}"
4,질병사망,2026-03-01,1,0.00178,MC000002,1,MC000003,19,NaN,NaN,...,NaN,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 19), (MC000002, 1)}"


In [8]:
df_joined["위험률조건_개발원"] = df_joined['위험률조건_회사']\
    .apply(lambda iters: 
        set(
            (row['위험구분_개발원'], row['위험구분유효값_개발원'])
            for _, row in df_factor_map.iterrows()
            for (회사_구분자, 회사_유효값) in iters
            if row['위험구분_회사'] == 회사_구분자 and row['위험구분유효값_회사'] == 회사_유효값
        )
    )

print(df_joined.shape)
df_joined.head()

(528, 24)


,위험률명,이력일자,산출단위,위험률,구분자1,구분자1 유효값,구분자2,구분자2 유효값,구분자3,구분자3 유효값,...,구분자6,구분자6 유효값,위험률코드_개발원,위험구분1,위험구분2,위험구분3,위험구분4,위험률코드_회사,위험률조건_회사,위험률조건_개발원
0,질병사망,2026-03-01,1,0.00090,MC000002,1,MC000003,15,NaN,NaN,...,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 15), (MC000002, 1)}","{(50, 15), (49, 1)}"
1,질병사망,2026-03-01,1,0.00112,MC000002,1,MC000003,16,NaN,NaN,...,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 16), (MC000002, 1)}","{(50, 16), (49, 1)}"
2,질병사망,2026-03-01,1,0.00134,MC000002,1,MC000003,17,NaN,NaN,...,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 17), (MC000002, 1)}","{(49, 1), (50, 17)}"
3,질병사망,2026-03-01,1,0.00156,MC000002,1,MC000003,18,NaN,NaN,...,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 18), (MC000002, 1)}","{(50, 18), (49, 1)}"
4,질병사망,2026-03-01,1,0.00178,MC000002,1,MC000003,19,NaN,NaN,...,NaN,NaN,200000,49,50,NaN,NaN,MR000001,"{(MC000003, 19), (MC000002, 1)}","{(50, 19), (49, 1)}"


In [9]:
for n in tqdm(range(1, 5)):
    위험구분컬럼명 = f'위험구분{n}'
    위험구분유효값컬럼명 = f'위험구분{n} 유효값'
    
    df_joined[위험구분유효값컬럼명] = df_joined[[위험구분컬럼명, '위험률조건_개발원']]\
        .apply(lambda row: 
            next(
                (유효값 for (구분자, 유효값) in row['위험률조건_개발원'] if 구분자 == row[위험구분컬럼명]),
                np.nan
            ),
            axis=1
        )   
        
print(df_joined.shape)
df_joined.head()

100%|██████████| 4/4 [00:00<00:00, 214.31it/s]

(528, 28)


,위험률명,이력일자,산출단위,위험률,구분자1,구분자1 유효값,구분자2,구분자2 유효값,구분자3,구분자3 유효값,...,위험구분2,위험구분3,위험구분4,위험률코드_회사,위험률조건_회사,위험률조건_개발원,위험구분1 유효값,위험구분2 유효값,위험구분3 유효값,위험구분4 유효값
0,질병사망,2026-03-01,1,0.00090,MC000002,1,MC000003,15,NaN,NaN,...,50,NaN,NaN,MR000001,"{(MC000003, 15), (MC000002, 1)}","{(50, 15), (49, 1)}",1,15,NaN,NaN
1,질병사망,2026-03-01,1,0.00112,MC000002,1,MC000003,16,NaN,NaN,...,50,NaN,NaN,MR000001,"{(MC000003, 16), (MC000002, 1)}","{(50, 16), (49, 1)}",1,16,NaN,NaN
2,질병사망,2026-03-01,1,0.00134,MC000002,1,MC000003,17,NaN,NaN,...,50,NaN,NaN,MR000001,"{(MC000003, 17), (MC000002, 1)}","{(49, 1), (50, 17)}",1,17,NaN,NaN
3,질병사망,2026-03-01,1,0.00156,MC000002,1,MC000003,18,NaN,NaN,...,50,NaN,NaN,MR000001,"{(MC000003, 18), (MC000002, 1)}","{(50, 18), (49, 1)}",1,18,NaN,NaN
4,질병사망,2026-03-01,1,0.00178,MC000002,1,MC000003,19,NaN,NaN,...,50,NaN,NaN,MR000001,"{(MC000003, 19), (MC000002, 1)}","{(50, 19), (49, 1)}",1,19,NaN,NaN


결과 출력

In [10]:
value_cols = [f'위험구분{n} 유효값' for n in range(1, 5)]

df_result = df_joined[
    ['위험률코드_개발원', '위험률명', '이력일자', '산출단위', *value_cols, '위험률']
].copy()

df_result[value_cols] = (
    df_result[value_cols]
    .astype('string')
    .fillna('00')
    .apply(lambda col: col.str.zfill(2))
)

In [11]:
df_result.to_csv(os.path.join(BASE_DIR, "output.txt"), sep="\t", index=False, header=None)